In [1]:
   import os
   from dotenv import load_dotenv, find_dotenv

   print("Looking for .env at:", find_dotenv())
   load_dotenv(find_dotenv())
   API_KEY = os.getenv("OPENAQ_API_KEY")
   print("Key found:", API_KEY is not None)
   print("Key length:", len(API_KEY) if API_KEY else 0)

Looking for .env at: c:\Users\Administrator\Documents\aqi-forecast-delhi\.env
Key found: True
Key length: 64


In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
API_KEY = os.getenv("OPENAQ_API_KEY")

url = "https://api.openaq.org/v3/locations"
headers = {"X-API-Key": API_KEY}
params = {
    "coordinates": "28.6139,77.2090",   # latitude,longitude of central Delhi
    "radius": 25000,                     # search radius in metres (25 km)
    "parameters_id": 2,                  # 2 = PM2.5
    "limit": 100,                        # return up to 100 stations
}

response = requests.get(url, headers=headers, params=params)
print("Status:", response.status_code)
print("Stations found:", len(response.json().get("results", [])))

Status: 200
Stations found: 94


In [3]:
rows = []
for d in response.json()["results"]:
    rows.append({
        "id": d["id"],
        "name": d["name"],
        "lat": d["coordinates"]["latitude"],
        "lon": d["coordinates"]["longitude"],
        "first": (d.get("datetimeFirst") or {}).get("utc"),
        "last": (d.get("datetimeLast") or {}).get("utc"),
    })

stations = pd.DataFrame(rows)
stations = stations.sort_values("first")
stations

,id,name,lat,lon,first,last
11,2597,US Diplomatic Post: New Delhi,28.598096,77.189066,2016-01-30T00:30:00Z,2016-11-09T16:30:00Z
6,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032,2016-02-05T14:15:00Z,2026-10-13T18:00:00Z
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,2016-02-05T14:55:00Z,2026-10-13T18:00:00Z
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,2016-02-05T14:55:00Z,2026-10-13T18:00:00Z
7,236,"Mandir Marg, Delhi - DPCC",28.634100,77.200500,2016-02-05T15:00:00Z,2018-02-21T21:15:00Z
...,...,...,...,...,...,...
80,3409496,"New Moti Bagh, Delhi - MHUA",28.578340,77.184000,2026-03-10T10:30:00Z,2026-06-13T09:15:00Z
92,6299494,"Prashant Garden, Khora - UPPCB",28.611190,77.342060,2026-04-07T12:45:00Z,2026-10-13T18:00:00Z
93,6299678,"Ved Vihar-Loni, Ghaziabad - UPPCB",28.739153,77.273668,2026-04-07T17:00:00Z,2026-10-13T18:00:00Z
1,15,IGI Airport,28.560000,77.094000,NaN,NaN
